# 03 - Populate and test the index

Upload `output/embedded_chunks.json` from notebook 01 into the index from notebook 02, then try a few search calls and inspect the results. No evaluation suite or assertion checks.

In [ ]:
import json
import os
from pathlib import Path
import requests
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, ClientSecretCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

ROOT = Path.cwd() if (Path.cwd() / 'samples').is_dir() else Path.cwd().parent
load_dotenv(ROOT / '.env')
credential = AzureCliCredential(subscription=os.getenv('AZURE_SUBSCRIPTION_ID') or None, process_timeout=60)
search_client = SearchClient(
    endpoint=os.environ['SEARCH_ENDPOINT'], index_name=os.getenv('SEARCH_INDEX', 'hr-demo'), credential=credential,
)
documents = json.loads((ROOT / 'output' / 'embedded_chunks.json').read_text(encoding='utf-8'))
print('Chunks to upload:', len(documents))


## Populate the index

Upload in small batches using the sample's `upload_documents` call. Show failures directly; SDK exceptions remain visible in the cell. Indexing can take a few seconds, so rerun a search cell if it is not ready yet.

In [ ]:
for offset in range(0, len(documents), 100):
    results = search_client.upload_documents(documents=documents[offset:offset + 100])
    print('Indexed:', sum(result.succeeded for result in results), 'Failed:', sum(not result.succeeded for result in results))
    for result in results:
        if not result.succeeded:
            print(result.key, result.status_code, result.error_message)


## Try a query

The persona comes from an identity, the way the agent does it: the cell gets a Microsoft Entra token for mock Workday, Workday reads who you are from that token and says whether you're a manager, and the answer becomes the security filter. No keys: Search takes your Azure CLI sign-in and Workday takes the token. Use your own sign-in, or add a service principal per role you want to test. Change a role on the fly with `mock_workday\set-role.ps1` and rerun the cell.

This is the [security filter pattern](https://learn.microsoft.com/azure/search/search-security-trimming-for-azure-search): string matching in an OData filter, not an authorization check. Whatever calls Search has to resolve the caller's groups first and is trusted to pass them honestly.

In [ ]:
query = 'How do I prepare for a performance calibration?'
# Also try:
#   'How many hours of paid sick leave can I carry over?'  (answer lives in several documents)
#   'How do I reset my Workday password?'                  (scripted answer)

# Who is asking: your own sign-in, or a service principal per role you want to test.
# Search itself always runs as the notebook's credential, like the agent's maker connection.
# A service principal needs a secret or certificate; your own sign-in and set-role.ps1 need neither.
personas = {
    'me': credential,
    # 'sp-manager': ClientSecretCredential('<tenant-id>', '<client-id>', '<client-secret>'),
    # 'sp-employee': ClientSecretCredential('<tenant-id>', '<client-id>', '<client-secret>'),
}
persona = 'me'

# Same steps as the agent: call (mock) Workday with the caller's Entra token, so Workday
# knows who is asking and says whether they are a manager; that becomes the security filter.
token = personas[persona].get_token(f"api://{os.environ['WORKDAY_API_APP_ID']}/.default").token
worker = requests.get(os.environ['WORKDAY_MOCK_URL'], headers={'Authorization': f'Bearer {token}'})
is_manager = worker.ok and worker.json()['isManager']
groups = ['all-employees', 'managers'] if is_manager else ['all-employees']

filter_text = f"group_ids/any(g: search.in(g, '{'|'.join(groups)}', '|'))"
select_fields = ['document_id', 'title', 'section_title', 'content', 'url',
                 'metadata_storage_path', 'persona', 'content_source']

def show_results(results):
    for result in results:
        print(result['document_id'], result['title'], '|', result['section_title'])
        print('Source:', result['content_source'], '| Persona:', result['persona'])
        print('Score:', result['@search.score'], 'Reranker:', result.get('@search.reranker_score'))
        print('Citation:', result['url'])
        print(result['content'][:500], '\n')

print('Workday', worker.status_code, worker.text, '\n', 'manager' if is_manager else 'employee', '->', filter_text)


### Keyword search

In [ ]:
results = search_client.search(search_text=query, filter=filter_text, select=select_fields, top=5)
show_results(results)


### Vector search

In [ ]:
vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=50, fields='content_vector')
results = search_client.search(
    search_text=None, vector_queries=[vector_query], vector_filter_mode='preFilter',
    filter=filter_text, select=select_fields, top=5,
)
show_results(results)


### Hybrid search with semantic reranking

Semantic ranker must be enabled on the Search service. The query vectorizer uses the same embedding deployment as notebook 01.

In [ ]:
results = search_client.search(
    search_text=query, vector_queries=[vector_query], vector_filter_mode='preFilter',
    query_type='semantic', semantic_configuration_name='hr-semantic',
    filter=filter_text, select=select_fields, top=5,
)
show_results(results)
